# Notebook 2: Decisions in Neo4j

This notebook loads the transactions from notebook 1 into Neo4j, gets a decision for each one and stores it as a `Decision` node linked to its transaction. After it runs, every question we want to ask in notebook 3 is a query.

The graph model is:

```
(:Transaction {txn_id, amount, merchant_category, ..., scenario, true_p, is_fraud})
  -[:HAS_DECISION]->
(:Decision {decision_id, choice, confidence, score, p_flag, p_pass, config_hash})
```

A few design choices are worth knowing about:

- **One decision per transaction per configuration.** The `config_hash` is a fingerprint of everything that shapes a decision -- the model, the question, the options and the fields Jev sees. Change any of them and the hash changes, so old decisions are never mistaken for new ones. The hash identifies a configuration, not an execution: running the same configuration again finds every transaction already decided and stores nothing new.
- **Safe to re-run.** Only transactions without a decision for the current configuration are sent to Jev, so an interrupted run picks up where it stopped.
- **The labels stay hidden.** The `scenario`, `true_p` and `is_fraud` properties live on the `Transaction` node so notebook 3 can score the decisions. They're never part of a request.
- **What's stored.** `p_flag` and `p_pass` are Jev's probabilities for each option. `confidence` is the margin between them, so the probability of the chosen option is `(1 + confidence) / 2`. `score` is Jev's self-rating, a continuous value between 0 and 2 that averages its three confidence levels.

## Two ways to run it

The `USE_SHIPPED_DECISIONS` switch in the Configuration cell decides where the decisions come from:

- **`True` (the default)** -- import `data/decisions.csv` into Neo4j. No Jev key is needed, and everyone who runs the notebook analyzes the same stored run.
- **`False`** -- call Jev for every transaction and export the results to `data/decisions.csv`. This needs a TypeSafe AI key.

## Prerequisites

Set the following environment variables before launching Jupyter:

```shell
export NEO4J_URI="neo4j+s://xxxx.databases.neo4j.io"
export NEO4J_USERNAME="your-username"
export NEO4J_PASSWORD="your-password"
export NEO4J_DATABASE="your-database"
export TYPESAFE_API_KEY="your-api-key"   # only if USE_SHIPPED_DECISIONS is False
```

Run notebook 1 first, or use the `data/transactions.csv` that ships with the notebooks.

## 1. Install Dependencies

In [1]:
%pip install neo4j==5.28.1 \
             numpy==2.1.3 \
             pandas==2.2.3 \
             requests==2.32.3 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import hashlib
import json
import os
import pandas as pd
import requests
import threading
import time

from concurrent.futures import ThreadPoolExecutor, as_completed
from neo4j import GraphDatabase
from pathlib import Path

## 3. Configuration

Two switches control the run. Read them before you execute anything below:

- `USE_SHIPPED_DECISIONS` -- `True` imports stored decisions, `False` calls Jev.
- `RESET_DATABASE` -- `True` wipes the database before loading. A free Aura instance holds one database, so a fresh start keeps old data from polluting the study.

The Jev settings must match notebook 1.

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]

print("Credentials set.")

Credentials set.


In [4]:
USE_SHIPPED_DECISIONS = True    # True: import data/decisions.csv   False: call Jev
RESET_DATABASE        = True    # True: wipe the database before loading

API_URL = "https://api.typesafe.ai/v1/systemone"
MODEL   = "jev-1.13.0"

STATE_FIELDS = [
    "amount", "merchant_category", "distance_from_home_km", "hour_of_day",
    "account_age_days", "prior_flags", "txns_last_24h", "typical_monthly_spend",
]
STATE_FORMAT_VERSION = 1   # bump if the way the state is built changes

DECISION_INSTRUCTIONS = (
    "Based on all available signals, should this transaction be flagged "
    "for review or passed through?"
)
DECISION_OPTIONS = {
    "Flag": "Potential fraud. Send for review.",
    "Pass": "Likely legitimate. Process normally.",
}
# Jev rates itself against these three levels (0, 1 and 2) and returns a continuous
# score between 0 and 2, the probability-weighted average of the levels.
SELF_RATING_INSTRUCTIONS = "How confident are you in this decision?"
SELF_RATING_LEVELS = ["Not confident", "Somewhat confident", "Very confident"]

# Request handling
WORKERS                  = 4
CONNECT_TIMEOUT          = 5
READ_TIMEOUT             = 30
MAX_RETRIES              = 3
MAX_CONSECUTIVE_FAILURES = 5

CHUNK_SIZE  = 100   # rows per UNWIND batch when loading transactions and importing decisions
WRITE_BATCH = 25    # decisions per write

print(f"USE_SHIPPED_DECISIONS = {USE_SHIPPED_DECISIONS}")
print(f"RESET_DATABASE        = {RESET_DATABASE}")
print("Configuration set.")

USE_SHIPPED_DECISIONS = True
RESET_DATABASE        = True
Configuration set.


## 4. Connect to Neo4j

In [5]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD),
    notifications_disabled_categories=["UNRECOGNIZED"]
)

print(driver.verify_connectivity())  # None is expected
print("Connection created.")

None
Connection created.


## 5. Reset the Database

**WARNING: the next cell deletes everything in the database it connects to.** With `RESET_DATABASE = True` (the default) it removes every node, relationship, constraint and index. Point this notebook at a database you don't mind clearing, or set `RESET_DATABASE = False` first.

When `RESET_DATABASE` is `True`, we remove everything: all constraints, all indexes (except the built-in lookup indexes, which Neo4j doesn't let you drop) and every node and relationship. The cell prints what's in the database before and after, so a wipe never comes as a surprise -- including any `Decision` nodes from an earlier run.

If you fetched decisions with a live key, the export step at the end of this notebook saves them to `data/decisions.csv`. A wiped database can then be refilled for free by setting `USE_SHIPPED_DECISIONS = True`.

In [6]:
def clear_database(session):
    # Drop all constraints
    constraints = session.run("SHOW CONSTRAINTS YIELD name RETURN name").data()
    for c in constraints:
        session.run(f"DROP CONSTRAINT {c['name']} IF EXISTS")

    # Drop all non-lookup indexes
    indexes = session.run("""
        SHOW INDEXES YIELD name, type
        WHERE type <> 'LOOKUP'
        RETURN name
    """).data()
    for idx in indexes:
        session.run(f"DROP INDEX {idx['name']} IF EXISTS")

    # Delete all nodes and relationships in batches
    while True:
        result = session.run("""
            MATCH (n) WITH n LIMIT 1000
            DETACH DELETE n
            RETURN count(*) AS deleted
        """)
        if result.single()["deleted"] == 0:
            break

    print("Database cleared -- constraints, indexes and data removed.")

def describe_database(session, title):
    rows = session.run("""
        MATCH (n)
        RETURN labels(n) AS labels, count(*) AS nodes
        ORDER BY nodes DESC
    """).data()
    print(title)
    if not rows:
        print("  Database is empty.")
    for r in rows:
        label = ":".join(r["labels"]) if r["labels"] else "(no label)"
        print(f"  {label}: {r['nodes']:,} nodes")

scheme, host = NEO4J_URI.split("://", 1)
print(f"Target: {scheme}://****.{host.split('.', 1)[-1]} (instance and database names hidden)")
print(f"RESET_DATABASE = {RESET_DATABASE}")

with driver.session(database=NEO4J_DATABASE) as session:
    describe_database(session, "Before:")
    if RESET_DATABASE:
        clear_database(session)
        describe_database(session, "After:")
    else:
        print("RESET_DATABASE is False -- existing data left in place.")

Target: neo4j+s://****.databases.neo4j.io (instance and database names hidden)
RESET_DATABASE = True
Before:
  Transaction: 500 nodes
  Decision: 500 nodes
Database cleared -- constraints, indexes and data removed.
After:
  Database is empty.


## 6. Load the Transactions

We read the CSV from notebook 1 and load it in batches with `UNWIND` and `MERGE`, so a partial re-run never creates duplicates. Two uniqueness constraints come first: one on `Transaction.txn_id` and one on `Decision.decision_id`, which is the transaction ID joined to the configuration hash.

In [7]:
transactions_df = pd.read_csv("data/transactions.csv")
records = transactions_df.to_dict("records")

print(f"Rows:  {len(transactions_df):,}")
print(f"Fraud: {transactions_df['is_fraud'].sum()} (hidden from Jev)")
transactions_df.head()

Rows:  500
Fraud: 189 (hidden from Jev)


,txn_id,amount,merchant_category,distance_from_home_km,hour_of_day,account_age_days,prior_flags,txns_last_24h,typical_monthly_spend,scenario,true_p,is_fraud
0,T0001,2992.06,travel,8.1,14,1483,0,2,3230,big_purchase,0.18,0
1,T0002,11.08,transport,4.8,13,1760,0,2,2710,routine,0.01,0
2,T0003,57.01,fuel,3.3,16,1120,0,1,4620,routine,0.01,0
3,T0004,173.97,transport,0.0,21,792,0,1,3070,routine,0.01,0
4,T0005,551.89,atm_withdrawal,204.9,0,673,0,2,3880,night_atm,0.40,0


In [8]:
def setup(session):
    session.run("""
        CREATE CONSTRAINT txn_id IF NOT EXISTS
        FOR (t:Transaction) REQUIRE t.txn_id IS UNIQUE
    """)
    session.run("""
        CREATE CONSTRAINT decision_id IF NOT EXISTS
        FOR (d:Decision) REQUIRE d.decision_id IS UNIQUE
    """)

    rows = session.run("""
        SHOW CONSTRAINTS YIELD name, labelsOrTypes, properties
        RETURN name, labelsOrTypes, properties
        ORDER BY name
    """).data()
    print("Constraints in place:")
    for r in rows:
        print(f"  {r['name']} on {r['labelsOrTypes'][0]}.{r['properties'][0]}")

def load_transactions(session, rows):
    result = session.run("""
        UNWIND $rows AS row
        MERGE (t:Transaction {txn_id: row.txn_id})
        SET t += row
    """, rows=rows)
    return result.consume().counters

with driver.session(database=NEO4J_DATABASE) as session:
    setup(session)

    chunks = [records[i:i + CHUNK_SIZE] for i in range(0, len(records), CHUNK_SIZE)]
    nodes_created = properties_set = 0
    print()
    for i, chunk in enumerate(chunks, start=1):
        counters = load_transactions(session, chunk)
        nodes_created  += counters.nodes_created
        properties_set += counters.properties_set
        print(f"  Batch {i} of {len(chunks)}: {counters.nodes_created} Transaction nodes created, "
              f"{counters.properties_set:,} properties set")

print(f"\nLoaded {len(records):,} transactions "
      f"({nodes_created:,} nodes created, {properties_set:,} properties set).")

Constraints in place:
  decision_id on Decision.decision_id
  txn_id on Transaction.txn_id

  Batch 1 of 5: 100 Transaction nodes created, 1,300 properties set
  Batch 2 of 5: 100 Transaction nodes created, 1,300 properties set
  Batch 3 of 5: 100 Transaction nodes created, 1,300 properties set
  Batch 4 of 5: 100 Transaction nodes created, 1,300 properties set
  Batch 5 of 5: 100 Transaction nodes created, 1,300 properties set

Loaded 500 transactions (500 nodes created, 6,500 properties set).


In [9]:
with driver.session(database=NEO4J_DATABASE) as session:
    r = session.run("""
        MATCH (t:Transaction)
        RETURN count(t) AS n, sum(t.is_fraud) AS fraud
    """).single()
    print(f"Transaction nodes: {r['n']:,}")
    print(f"Labeled fraud:     {r['fraud']}")

Transaction nodes: 500
Labeled fraud:     189


## 7. Decision Helpers

These functions build the request, call Jev and flatten the response. Defining them needs no key, so the cell runs either way.

`parse_response` always returns the same keys, with `None` where Jev didn't send a value, so every `Decision` node has the same shape. It raises an error if Jev returns an option we didn't offer.

In [10]:
class JevError(Exception):
    pass

def build_state(txn):
    """Visible features only. Hidden fields never reach the model."""
    return {field: txn[field] for field in STATE_FIELDS}

def build_body(state):
    return {
        "model": MODEL,
        "state": json.dumps(state, sort_keys=True),
        "questions": {
            "decision": {
                "type": "choice",
                "instructions": DECISION_INSTRUCTIONS,
                "criteria": DECISION_OPTIONS,
            },
            "self_rating": {
                "type": "score",
                "instructions": SELF_RATING_INSTRUCTIONS,
                "criteria": SELF_RATING_LEVELS,
            },
        },
    }

def config_hash():
    """Fingerprint of everything that shapes a decision."""
    spec = {
        "model": MODEL,
        "decision_instructions": DECISION_INSTRUCTIONS,
        "decision_options": DECISION_OPTIONS,
        "self_rating_instructions": SELF_RATING_INSTRUCTIONS,
        "self_rating_levels": SELF_RATING_LEVELS,
        "state_fields": STATE_FIELDS,
        "state_format_version": STATE_FORMAT_VERSION,
    }
    blob = json.dumps(spec, sort_keys=True).encode()
    return hashlib.sha256(blob).hexdigest()[:16]

def to_float(value):
    return None if value is None else float(value)

def parse_response(payload):
    answers  = payload.get("answers") or {}
    decision = answers.get("decision") or {}
    rating   = answers.get("self_rating") or {}
    probs    = decision.get("probabilities") or {}

    choice = decision.get("choice")
    if choice not in DECISION_OPTIONS:
        raise JevError(f"Unexpected choice {choice!r} in response")

    return {
        "choice":        choice,
        "confidence":    to_float(decision.get("confidence")),
        "score":         to_float(rating.get("score")),
        "p_flag":        to_float(probs.get("Flag")),
        "p_pass":        to_float(probs.get("Pass")),
        "probabilities": json.dumps(probs, sort_keys=True),
    }

_local = threading.local()

def http_session():
    if not hasattr(_local, "session"):
        _local.session = requests.Session()
    return _local.session

def decide(state, api_key):
    """One call. Retries on timeouts, 429 and 5xx. Fails fast on other 4xx."""
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    body    = build_body(state)
    timeout = (CONNECT_TIMEOUT, READ_TIMEOUT)

    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        started = time.perf_counter()
        try:
            response = http_session().post(API_URL, headers=headers, json=body, timeout=timeout)
        except requests.RequestException as exc:
            last_error = f"request failed: {exc}"
        else:
            if response.status_code == 200:
                result = parse_response(response.json())
                result["latency_ms"] = round((time.perf_counter() - started) * 1000, 1)
                return result
            if response.status_code != 429 and response.status_code < 500:
                raise JevError(f"API error {response.status_code}: {response.text[:300]}")
            last_error = f"API error {response.status_code}"

        if attempt < MAX_RETRIES:
            time.sleep(0.5 * 2 ** attempt)

    raise JevError(f"Gave up after {MAX_RETRIES + 1} attempts ({last_error})")

CURRENT_HASH = config_hash()
print(f"Configuration hash: {CURRENT_HASH}")

Configuration hash: aae40cf18cbd071a


## 8. Store the Decisions

Both routes write through the same query, so a decision looks identical in Neo4j whether it came from Jev or from the CSV.

Repeating a write never adds a second copy. The uniqueness constraint on `decision_id` and the `MERGE` on that key mean a repeat updates the same node, and the second `MERGE` does the same for the `HAS_DECISION` relationship. The `created_at` timestamp is set with `ON CREATE`, so it records when the decision was first stored and a repeat doesn't overwrite it.

In [11]:
WRITE_QUERY = """
UNWIND $rows AS row
MATCH (t:Transaction {txn_id: row.txn_id})
MERGE (d:Decision {decision_id: row.decision_id})
  ON CREATE SET d.created_at = datetime()
SET d += row.props
MERGE (t)-[:HAS_DECISION]->(d)
"""

def write_decisions(session, results, cfg_hash):
    """results: list of (txn_id, props) pairs."""
    rows = [
        {
            "txn_id":      txn_id,
            "decision_id": f"{txn_id}|{cfg_hash}",
            "props":       {**props, "txn_id": txn_id, "config_hash": cfg_hash},
        }
        for txn_id, props in results
    ]
    return session.run(WRITE_QUERY, rows=rows).consume().counters

### Route A: import the shipped decisions

Used when `USE_SHIPPED_DECISIONS` is `True`. The CSV carries its own `config_hash`, so the imported decisions keep the fingerprint of the run that produced them.

In [12]:
DECISION_COLUMNS = [
    "choice", "confidence", "score", "p_flag", "p_pass", "probabilities", "latency_ms",
]

if USE_SHIPPED_DECISIONS:
    path = Path("data/decisions.csv")
    if not path.exists():
        raise FileNotFoundError(
            "data/decisions.csv not found. Add the shipped file, or set "
            "USE_SHIPPED_DECISIONS = False and run with a TypeSafe AI key."
        )

    shipped_df = pd.read_csv(path)
    shipped_df = shipped_df.astype(object).where(shipped_df.notna(), None)

    nodes_created = relationships_created = 0
    with driver.session(database=NEO4J_DATABASE) as session:
        for cfg_hash, group in shipped_df.groupby("config_hash"):
            results = [
                (row["txn_id"], {col: row[col] for col in DECISION_COLUMNS})
                for row in group.to_dict("records")
            ]
            batches = [results[i:i + CHUNK_SIZE] for i in range(0, len(results), CHUNK_SIZE)]
            print(f"Configuration {cfg_hash}:")
            for i, batch in enumerate(batches, start=1):
                counters = write_decisions(session, batch, cfg_hash)
                nodes_created         += counters.nodes_created
                relationships_created += counters.relationships_created
                print(f"  Batch {i} of {len(batches)}: {counters.nodes_created} Decision nodes created, "
                      f"{counters.relationships_created} relationships created")

    print(f"\nImported {len(shipped_df):,} decisions from {path} "
          f"({nodes_created:,} nodes and {relationships_created:,} relationships created).")
else:
    print("USE_SHIPPED_DECISIONS is False -- skipping the import.")

Configuration aae40cf18cbd071a:
  Batch 1 of 5: 100 Decision nodes created, 100 relationships created
  Batch 2 of 5: 100 Decision nodes created, 100 relationships created
  Batch 3 of 5: 100 Decision nodes created, 100 relationships created
  Batch 4 of 5: 100 Decision nodes created, 100 relationships created
  Batch 5 of 5: 100 Decision nodes created, 100 relationships created

Imported 500 decisions from data/decisions.csv (500 nodes and 500 relationships created).


### Route B: call Jev

Used when `USE_SHIPPED_DECISIONS` is `False`. We ask Neo4j which transactions have no decision for the current configuration, then send them to Jev with a few workers in parallel. Results are written in small batches, so an interruption loses at most one batch. If several calls fail in a row, the run stops instead of burning through the rest -- run the cell again to retry.

In [13]:
PENDING_QUERY = """
MATCH (t:Transaction)
WHERE NOT EXISTS { (t)-[:HAS_DECISION]->(:Decision {config_hash: $hash}) }
RETURN t {.*} AS txn
ORDER BY t.txn_id
"""

def decide_one(txn, api_key):
    return txn["txn_id"], decide(build_state(txn), api_key)

def run_decisions(session, api_key):
    pending = [rec["txn"] for rec in session.run(PENDING_QUERY, hash=CURRENT_HASH)]
    print(f"{len(pending)} transactions need a decision (configuration {CURRENT_HASH}).")
    if not pending:
        return

    buffer, done, failures, consecutive = [], 0, 0, 0
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = [pool.submit(decide_one, txn, api_key) for txn in pending]
        try:
            for future in as_completed(futures):
                try:
                    txn_id, result = future.result()
                except JevError as exc:
                    failures += 1
                    consecutive += 1
                    print(f"  failed: {exc}")
                    if consecutive >= MAX_CONSECUTIVE_FAILURES:
                        print("Too many consecutive failures -- stopping.")
                        for f in futures:
                            f.cancel()
                        break
                    continue
                consecutive = 0
                done += 1
                buffer.append((txn_id, result))
                if len(buffer) >= WRITE_BATCH:
                    write_decisions(session, buffer, CURRENT_HASH)
                    buffer.clear()
                    print(f"  {done} of {len(pending)} decisions stored")
        finally:
            if buffer:
                write_decisions(session, buffer, CURRENT_HASH)

    print(f"\nStored {done} decisions, {failures} failures.")
    if failures:
        print("Run this cell again to retry the failures.")

if not USE_SHIPPED_DECISIONS:
    api_key = os.environ["TYPESAFE_API_KEY"]
    with driver.session(database=NEO4J_DATABASE) as session:
        run_decisions(session, api_key)
else:
    print("USE_SHIPPED_DECISIONS is True -- no calls made.")

USE_SHIPPED_DECISIONS is True -- no calls made.


## 9. Verify

Both routes should leave one `Decision` per transaction for each configuration, linked by `HAS_DECISION`.

In [14]:
with driver.session(database=NEO4J_DATABASE) as session:
    r = session.run("MATCH (d:Decision) RETURN count(d) AS n").single()
    print(f"Decision nodes:    {r['n']:,}")
    r = session.run("MATCH ()-[:HAS_DECISION]->() RETURN count(*) AS n").single()
    print(f"HAS_DECISION rels: {r['n']:,}")

    rows = session.run("""
        MATCH (d:Decision)
        RETURN d.config_hash AS config_hash,
               d.choice      AS choice,
               count(*)      AS decisions
        ORDER BY config_hash, choice
    """).data()

pd.DataFrame(rows)

Decision nodes:    500
HAS_DECISION rels: 500


,config_hash,choice,decisions
0,aae40cf18cbd071a,Flag,253
1,aae40cf18cbd071a,Pass,247


## 10. Export

After a live run we export the decisions to a CSV file. A `decisions.csv` in the `data/` folder is what makes the study reproducible without a key: anyone can use it with `USE_SHIPPED_DECISIONS = True` and analyze the same stored run.

If `data/decisions.csv` already exists, which it does when you've cloned the repository, the export goes to a new timestamped file such as `data/decisions_live_20260101_120000.csv`. That way the shipped run behind the article's numbers is never overwritten. To use your own run later, swap the new file in for `decisions.csv`.

When the decisions were imported, nothing is exported.

In [15]:
EXPORT_COLUMNS = [
    "txn_id", "config_hash", "choice", "confidence", "score",
    "p_flag", "p_pass", "probabilities", "latency_ms",
]

EXPORT_QUERY = """
MATCH (t:Transaction)-[:HAS_DECISION]->(d:Decision {config_hash: $hash})
RETURN t.txn_id        AS txn_id,
       d.config_hash   AS config_hash,
       d.choice        AS choice,
       d.confidence    AS confidence,
       d.score         AS score,
       d.p_flag        AS p_flag,
       d.p_pass        AS p_pass,
       d.probabilities AS probabilities,
       d.latency_ms    AS latency_ms
ORDER BY t.txn_id
"""

if not USE_SHIPPED_DECISIONS:
    with driver.session(database=NEO4J_DATABASE) as session:
        export_df = pd.DataFrame(session.run(EXPORT_QUERY, hash=CURRENT_HASH).data())
    export_df = export_df[EXPORT_COLUMNS]
    Path("data").mkdir(exist_ok=True)
    export_path = Path("data/decisions.csv")
    if export_path.exists():
        # Never overwrite the shipped run that the article's numbers come from
        export_path = Path(f"data/decisions_live_{time.strftime('%Y%m%d_%H%M%S')}.csv")
    export_df.to_csv(export_path, index=False)
    print(f"Exported {len(export_df):,} decisions to {export_path}")
    if len(export_df) < len(transactions_df):
        print(f"Warning: {len(transactions_df) - len(export_df)} transactions have no decision yet.")
        print("Run the Route B cell again, then re-run this cell, before sharing the file.")
else:
    print("USE_SHIPPED_DECISIONS is True -- nothing to export.")

USE_SHIPPED_DECISIONS is True -- nothing to export.


## 11. Teardown

In [16]:
driver.close()
print("Connection closed.")

Connection closed.
